In [1]:
import numpy as np
import scipy.optimize as optimize
from scipy.stats import gamma

In [10]:
from GASModels.components.trend import Trend, TrendTypes
from GASModels.dynamics import Dynamics

In [11]:
args = [0.00, 0.02]
components = [Trend(TrendTypes.RANDOM_WALK, args[1])]
# Try different initialization approaches
dynamics = Dynamics(10, [0], components, args)
dynamics.iterate()


TypeError: Trend.include_dynamics() missing 1 required positional argument: 'score'

In [6]:
def quadratic(x):
    return x**2 + 4*x - 6

In [8]:
result = optimize.minimize(
            quadratic,
            [10],
            method='L-BFGS-B',
            options={'maxiter': 1000}
        )

In [9]:
result

  message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
  success: True
   status: 0
      fun: -9.999999999999954
        x: [-2.000e+00]
      nit: 2
      jac: [ 3.553e-07]
     nfev: 8
     njev: 4
 hess_inv: <1x1 LbfgsInvHessProduct with dtype=float64>

In [ ]:
import numpy as np
import scipy.optimize as optimize
from scipy.stats import gamma

class GammaGASModel:
    def __init__(self, y, harmonics):
        self.y = y
        self.n = len(y)
        self.harmonics = harmonics
        
    def model(self, params):
        # Extract parameters
        mu_init = params[0]
        gamma_init = params[1:13]  # 12 initial seasonal values
        m = params[13]
        k_mu = params[14]
        k_gamma = params[15]
        alpha = params[16]
        
        # Initialize arrays
        mu = np.zeros(self.n + 1)
        gamma_vals = np.zeros((self.n + 1, self.harmonics))
        gamma_star_vals = np.zeros((self.n + 1, self.harmonics))
        lambda_vals = np.zeros(self.n)
        loglik = np.zeros(self.n)
        
        # Set initial values
        mu[0] = mu_init
        gamma_vals[0, :] = gamma_init
        
        for t in range(2, self.n):
            # Calculate seasonal component using harmonics
            seasonal = 0
            for h in range(1, self.harmonics + 1):
                freq = 2 * np.pi * h / 12
                time_point = (t % 12) + 1
                gamma_vals[t, h] = 
            
            # Calculate lambda
            lambda_vals[t] = np.exp(mu[t] + seasonal)
            
            # Gamma distribution log-likelihood
            if self.y[t] > 0 and lambda_vals[t] > 0:
                shape_param = lambda_vals[t] / alpha
                loglik[t] = gamma.logpdf(self.y[t], shape_param, scale=alpha)
            else:
                loglik[t] = -1e10
            
            # Score (derivative of log-likelihood w.r.t. lambda)
            score = (self.y[t] - lambda_vals[t]) / (alpha * lambda_vals[t])
            
            # Update mu
            mu[t+1] = m + mu[t] + k_mu * score
            
            # Update seasonal components
            for h in range(1, self.harmonics + 1):
                freq = 2 * np.pi * h / 12
                time_point = (t % 12) + 1
                
                dseasonal_dcos = np.cos(freq * time_point)
                dseasonal_dsin = np.sin(freq * time_point)
                
                gamma_vals[t+1, h*2-2] = (gamma_vals[t, h*2-2] + 
                                         k_gamma * score * dseasonal_dcos)
                gamma_vals[t+1, h*2-1] = (gamma_vals[t, h*2-1] + 
                                         k_gamma * score * dseasonal_dsin)
        
        return -np.sum(loglik), mu, gamma_vals, lambda_vals
    
    def neg_loglikelihood(self, params):
        loglik, _, _, _ = self.model(params)
        return loglik
    
    def optimize(self):
        # Initial parameter guesses
        init_params = np.array([
            np.log(np.mean(self.y)),  # mu_init
            *np.zeros(12),           # gamma_init (12 zeros)
            0,                       # m
            0.1,                     # k_mu
            0.01,                    # k_gamma
            1.0                      # alpha
        ])
        
        # Parameter bounds
        bounds = [
            (-10, 10),               # mu_init
            *[(-5, 5) for _ in range(12)],  # gamma_init
            (-1, 1),                 # m
            (0, 1),                  # k_mu
            (0, 1),                  # k_gamma
            (0.001, 10)              # alpha
        ]
        
        # Optimize
        result = optimize.minimize(
            self.neg_loglikelihood,
            init_params,
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 1000}
        )
        
        return result

# Example usage
if __name__ == "__main__":
    # Generate sample data
    np.random.seed(123)
    n = 200
    y = np.random.gamma(shape=2, scale=2, size=n)
    
    # Fit model
    model = GammaGASModel(y)
    result = model.optimize()
    
    print("Optimized parameters:")
    param_names = ['mu_init'] + [f'gamma_init_{i}' for i in range(1, 13)] + ['m', 'k_mu', 'k_gamma', 'alpha']
    for name, value in zip(param_names, result.x):
        print(f"{name}: {value:.4f}")
    
    print(f"\nNegative log-likelihood: {result.fun:.4f}")